# KshetraAI: Training on Google Colab

This notebook runs the full KshetraAI pipeline on Colab. Use it when your local machine is slow or low on RAM.

**What this notebook does**

1. Mounts your Google Drive
2. Gets the KshetraAI code (either from a zip in Drive or from GitHub)
3. Installs dependencies
4. Copies the 8 raw Syngenta CSV files from Drive into the project
5. Runs the full pipeline (data load, features, train, score, route, anomaly, backtest)
6. Saves all outputs back to Drive so you can download them anywhere
7. Optional: runs a quick demo to see one territory's beat plan

**Before you start**

- Place the 8 raw Syngenta CSV files in a Drive folder, for example: `MyDrive/syngenta_data/`
- Either upload `KshetraAI.zip` to Drive, or push the code to a GitHub repo
- Set the runtime type to Python 3 (Runtime menu -> Change runtime type). GPU is not needed; LightGBM trains fast on CPU.

Expected total runtime on Colab free tier: 5 to 8 minutes.


## Step 1. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


## Step 2. Get the code

Pick **one** of the two methods below.

**Method A** is simplest: just unzip `KshetraAI.zip` from Drive. Use this for the hackathon.

**Method B** uses GitHub. Use this once you have set up a repo and want clean version control.


### Method A: Unzip from Drive (recommended for hackathon)

In [ ]:
# Update this path to wherever you uploaded KshetraAI.zip in Drive
ZIP_PATH = '/content/drive/MyDrive/KshetraAI.zip'

import os, shutil
if os.path.exists('/content/KshetraAI'):
    shutil.rmtree('/content/KshetraAI')

!cp "$ZIP_PATH" /content/
!unzip -q /content/KshetraAI.zip -d /content/
%cd /content/KshetraAI
!ls -la


### Method B: Clone from GitHub (alternative)

Uncomment and edit the URL if you are using GitHub. Skip this cell if you used Method A above.

In [ ]:
# Uncomment to use GitHub instead of zip
# REPO_URL = 'https://github.com/your-username/KshetraAI.git'
#
# import os, shutil
# if os.path.exists('/content/KshetraAI'):
#     shutil.rmtree('/content/KshetraAI')
# !git clone $REPO_URL /content/KshetraAI
# %cd /content/KshetraAI
# !ls -la


## Step 3. Install dependencies

In [ ]:
!pip install -q -r requirements.txt

# Verify everything imports cleanly
import lightgbm, shap, pandas, numpy
from ortools.constraint_solver import pywrapcp
print(f"lightgbm  {lightgbm.__version__}")
print(f"shap      {shap.__version__}")
print(f"pandas    {pandas.__version__}")
print(f"numpy     {numpy.__version__}")
print("ortools   OK")


## Step 4. Copy raw data from Drive

Place the 8 Syngenta CSVs in a Drive folder, then update `DATA_DIR` below to match.

In [ ]:
# Update this to your Drive folder containing the 8 raw CSVs
DATA_DIR = '/content/drive/MyDrive/syngenta_data'

!cp "$DATA_DIR"/*.csv raw_data/
!ls -la raw_data/

# Verify all 8 expected files are present
import os
expected = [
    'retailers.csv', 'reps_territory.csv', 'retailer_visit_log.csv',
    'retailer_inventory_weekly.csv', 'retailer_pos.csv', 'growers.csv',
    'digital_funnel_weekly.csv', 'whatsapp_campaign.csv'
]
missing = [f for f in expected if not os.path.exists(f'raw_data/{f}')]
if missing:
    print(f"MISSING FILES: {missing}")
else:
    print("All 8 raw files present")


## Step 5. Run the full pipeline

This is the main step. It will:

- Load and clean all 8 files
- Fetch weather from Open-Meteo (one time, then cached)
- Build labels and time-aware features
- Train the LambdaRank ranker with NDCG at 5
- Compute SHAP explanations for every retailer
- Optimise routes for all 500 territories
- Detect anomalies and produce the priority feed
- Backtest the ranking against the actual rep behaviour

Expected time on Colab free tier: 5 to 8 minutes.

In [ ]:
# Option A: use the synthetic data generator (no Syngenta data needed)
!python generate_data.py

# Option B: if you have the real 8 CSVs in Drive, comment the line above
# and uncomment the copy in the previous cell instead.

!python run_pipeline.py

# The pipeline now also writes:
#   outputs/evaluation_report.json   precision@k, faithfulness, acceptance
#   outputs/salah_samples.json       LLM next-best-action samples


## Step 6. Save outputs back to Drive

This copies the generated files into Drive so you can access them later from anywhere.

In [ ]:
# Update if you want a different Drive folder for outputs
OUTPUT_DIR = '/content/drive/MyDrive/KshetraAI_outputs'

!mkdir -p "$OUTPUT_DIR"
!cp -r outputs/* "$OUTPUT_DIR/"
!cp -r models/* "$OUTPUT_DIR/"
!cp -r processed/weather_data.csv "$OUTPUT_DIR/" 2>/dev/null || true
# Also copy the frontend bundle (HTML + data.js) so you can open the live
# dashboard from Drive on any machine
!mkdir -p "$OUTPUT_DIR/frontend"
!cp frontend/index.html "$OUTPUT_DIR/frontend/"
!cp frontend/data.js "$OUTPUT_DIR/frontend/" 2>/dev/null || true

!ls -la "$OUTPUT_DIR"
print(f"\nAll outputs saved to: {OUTPUT_DIR}")
print(f"Frontend with live data at: {OUTPUT_DIR}/frontend/index.html")


## Step 7 (optional). Quick demo: one territory's beat plan

This loads the saved model and produces a beat plan for one territory, including SHAP reasoning. Useful for verifying the pipeline produced what you expected, and for the live demo on presentation day.

In [ ]:
import pandas as pd
import sys
sys.path.insert(0, '/content/KshetraAI')

from src import config, data_io, weather as weather_mod, ranker
from src import features, scoring, route, explain, predict

# Load everything
data = data_io.load_all()
weather = weather_mod.get_weather()        # uses cache
model = ranker.load(f"{config.MODELS}/lambdarank.txt")

# Score every retailer
scored = predict.score_all(data['retailers'], data, weather, model)

# Pick the highest scoring territory for demo
top_territory = scored.groupby('territory_id')['final_score_100'].mean().idxmax()
print(f"\nDemo territory: {top_territory}\n")

# Generate SHAP explanations
expl = explain.build_explainer(model)
shap_df = explain.shap_table(expl, scored)

# Build the beat plan
plan = predict.beat_plan(top_territory, scored, model, expl, shap_df)

print(f"Territory: {plan['territory_id']}")
print(f"Stops: {len(plan['stops'])}")
print("-" * 60)
for stop in plan['stops']:
    print(f"\n  Stop {stop['visit_order']}: {stop['retailer_id']}")
    print(f"    Tehsil: {stop['tehsil']}")
    print(f"    Score:  {stop['final_score']:.1f} ({stop['tier']})")
    if 'why' in stop and stop['why']['reasons']:
        print(f"    Why:    {', '.join(stop['why']['reasons'])}")
    if 'why' in stop and stop['why']['watch']:
        print(f"    Watch:  {stop['why']['watch'][0]}")
print(f"\n  Total route: {plan['stops'][0]['route_km_indicative']:.1f} km (indicative)")


## What now

The outputs are in your Drive at `KshetraAI_outputs/`:

- `final_scores.csv` -- every retailer with rule, ml, and hybrid scores
- `shap_values.csv` -- SHAP contributions per retailer
- `optimized_routes.csv` -- ordered visit sequence per territory
- `anomaly_alerts.csv` -- priority ranked CHETAVANI feed
- `lambdarank.txt` -- trained model, ready to serve

Download these to your laptop and load them in VS Code for analysis or the demo dashboard.

**Iterating on the code**

If you change something locally and want to re-train on Colab:

- Method A: rebuild the zip locally, re-upload to Drive, restart this notebook from Step 2 onwards
- Method B: push to GitHub, then in this notebook run `!cd /content/KshetraAI && git pull && python run_pipeline.py`

Method B is faster once GitHub is set up.
